# Import data
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv
from langchain_postgres import PGEngine
from pydantic import SecretStr

dotenv.load_dotenv()
ENV_PG_CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")
ENV_ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# Obsidian Vault

In [ ]:
from sqlalchemy import create_engine
from agent_assistant.retriever.obsidian_llama import ObsidianLlamaRetriever

# エンジン初期化
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_ENV_GEMINI_API_KEY is not None

sa_engine = create_engine(ENV_PG_CONNECTION_STRING)
pg_engine = PGEngine.from_connection_string(ENV_PG_CONNECTION_STRING, pool_size=5)

In [ ]:
from pathlib import Path
from agent_assistant.loader.obsidian import VaultLoader, PgVault
from agent_assistant.model import ObsidianVaultBase

# Vault から読み込み
vault_path = Path("../docs/dataset_obsidian/")
loader = VaultLoader(vault_path)
docs = loader.load()

# DB へ取り込み
ObsidianVaultBase.metadata.create_all(sa_engine)
PgVault.sync(docs, sa_engine)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_ENV_GEMINI_API_KEY is not None

obsidian_retriever = ObsidianLlamaRetriever(
    sa_engine,
    ENV_PG_CONNECTION_STRING,
    SecretStr(ENV_ENV_GEMINI_API_KEY),
    "obsidian_vault_docstore",
    "obsidian_vault_vectors",
)

In [ ]:
# obsidian_retriever.sync_chunks()

In [ ]:
from llama_index.core.vector_stores.types import VectorStoreQuery, VectorStoreQueryMode
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

ember = GoogleGenAIEmbedding(api_key=os.getenv("ENV_GEMINI_API_KEY"))
query_res = obsidian_retriever._vector_store.query(
    VectorStoreQuery(
        query_embedding=ember.get_query_embedding("プロンプトエンジニアリング"),
        similarity_top_k=5,
        mode=VectorStoreQueryMode.MMR,
        mmr_threshold=0.5,
    )
)
assert query_res.nodes is not None
for item in query_res.nodes:
    print(item.text)  # pyright: ignore[reportAttributeAccessIssue]
    print("==================")